# R1A — Rust Object Prompt Generation

**Purpose:** Generate (Rust construct, Rust object) prompt variations with perplexity filter.
Mirrors 1A_object_prompts.ipynb exactly, adapted for Rust syntax.

**Validation:** tree-sitter-rust replaces Python's `ast` module.

**Stages:**
- A: Concept Matrix (construct families × object families + compatibility)
- B: Template Engine (RustPromptGenerator with per-construct lambdas)
- C: Perplexity Filter (same Qwen2.5-Coder-7B, same loss ranking)
- D: Pipeline (same checkpoint/parquet logic)

In [ ]:
# Cell 1 – Dependencies
import subprocess, sys, os

pkgs = ["transformers", "torch", "pyarrow", "accelerate",
        "tree-sitter", "tree-sitter-rust", "numpy", "pandas", "tqdm"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("Dependencies ready")

In [ ]:
# Cell 2 – Configuration
MODEL_ID = "Qwen/Qwen2.5-Coder-7B"

MODE = "test"  # "test" | "small" | "full"

LANG = "R"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
run_name = "R1A_object"
OUTPUT_FILE = f"{PREFIX}1A_object_prompts.parquet"
STATS_FILE = f"{PREFIX}1A_object_stats.json"

if MODE == "test":
    N_GENERATE = 10
    N_KEEP = 10
    CHECKPOINT_EVERY = 100
    CATASTROPHIC_THRESHOLD = 3.0
elif MODE == "small":
    N_GENERATE = 50
    N_KEEP = 50
    CHECKPOINT_EVERY = 100
    CATASTROPHIC_THRESHOLD = 3.0
elif MODE == "full":
    N_GENERATE = 100
    N_KEEP = 100
    CHECKPOINT_EVERY = 50
    CATASTROPHIC_THRESHOLD = 3.0

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

CHECKPOINT_DIR = DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Mode: {MODE} | N_GENERATE={N_GENERATE} N_KEEP={N_KEEP}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
# Cell 3 – Imports
import random
import json
import textwrap
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from datetime import datetime

import tree_sitter_rust as ts_rust
from tree_sitter import Language, Parser

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("R1A")

random.seed(42)
np.random.seed(42)

print("Imports OK")

In [ ]:
# Cell 4 – Google Drive mount (no-op if local)
# Already handled in Cell 2. This cell is a placeholder for Colab-specific setup.
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 5 – tree-sitter setup & validation helpers

RUST_LANGUAGE = Language(ts_rust.language())
ts_parser = Parser(RUST_LANGUAGE)


def parse_rust(code: str):
    """Parse Rust code and return tree-sitter tree."""
    return ts_parser.parse(bytes(code, "utf-8"))


def get_rust_node_types(code: str) -> set:
    """Walk tree-sitter tree and collect all node types."""
    tree = parse_rust(code)
    types = set()
    def walk(node):
        types.add(node.type)
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return types


def has_errors(code: str) -> bool:
    """Return True if tree-sitter found syntax errors."""
    tree = parse_rust(code)
    return tree.root_node.has_error


def _has_child_type(node, target_type: str) -> bool:
    """Check if node has any descendant of given type."""
    for child in node.children:
        if child.type == target_type:
            return True
        if _has_child_type(child, target_type):
            return True
    return False


def _find_nodes_of_type(root, target_type: str) -> list:
    """Find all nodes of a given type in the tree."""
    found = []
    def walk(node):
        if node.type == target_type:
            found.append(node)
        for child in node.children:
            walk(child)
    walk(root)
    return found


# Simple node concepts: concept name -> tree-sitter node type
SIMPLE_NODE_CONCEPTS = {
    "for_expression": "for_expression",
    "while_expression": "while_expression",
    "loop_expression": "loop_expression",
    "if_expression": "if_expression",
    "match_expression": "match_expression",
    "function_item": "function_item",
    "closure_expression": "closure_expression",
    "let_declaration": "let_declaration",
    "const_item": "const_item",
    "static_item": "static_item",
    "struct_item": "struct_item",
    "enum_item": "enum_item",
    "type_alias": "type_item",
    "impl_item": "impl_item",
    "trait_item": "trait_item",
    "use_declaration": "use_declaration",
    "mod_item": "mod_item",
    "return_expression": "return_expression",
    "break_expression": "break_expression",
    "continue_expression": "continue_expression",
    "async_block": "async_block",
    "await_expression": "await_expression",
    "unsafe_block": "unsafe_block",
    "reference_expression": "reference_expression",
    "lifetime": "lifetime",
    "macro_invocation": "macro_invocation",
    "attribute_item": "attribute_item",
    "dereference": "unary_expression",  # *expr
}


def verify_concept(code: str, concept: str) -> bool:
    """Verify that the given Rust concept appears in the code."""
    tree = parse_rust(code)
    root = tree.root_node

    if concept == "let_mut":
        # let_declaration with mutable_specifier child
        lets = _find_nodes_of_type(root, "let_declaration")
        return any(_has_child_type(n, "mutable_specifier") for n in lets)

    elif concept == "mutable_reference":
        # reference_expression with mutable_specifier child
        refs = _find_nodes_of_type(root, "reference_expression")
        return any(_has_child_type(n, "mutable_specifier") for n in refs)

    elif concept == "macro_invocation_question_mark":
        # The ? operator → try_expression
        return len(_find_nodes_of_type(root, "try_expression")) > 0

    elif concept in SIMPLE_NODE_CONCEPTS:
        target_type = SIMPLE_NODE_CONCEPTS[concept]
        return len(_find_nodes_of_type(root, target_type)) > 0

    return False


# Smoke test
test_cases = [
    ("fn main() { for i in 0..10 { println!(\"{}\", i); } }", "for_expression"),
    ("fn main() { let mut x = 5; x += 1; }", "let_mut"),
    ("fn main() { let x = 5; let y = &mut x; }", "mutable_reference"),
    ("struct Foo { x: i32 }", "struct_item"),
    ("fn main() { if true { println!(\"yes\"); } }", "if_expression"),
    ("fn main() { match 1 { 1 => (), _ => () } }", "match_expression"),
    ("fn foo() -> i32 { 42 }", "function_item"),
    ("fn main() { let c = |x: i32| x + 1; }", "closure_expression"),
    ("enum Color { Red, Green, Blue }", "enum_item"),
    ("use std::collections::HashMap;", "use_declaration"),
    ("#[derive(Debug)]\nstruct Foo;", "attribute_item"),
    ("fn main() { unsafe { let x = 42; } }", "unsafe_block"),
]

for code, concept in test_cases:
    result = verify_concept(code, concept)
    err = has_errors(code)
    status = "OK" if result and not err else "FAIL"
    print(f"  {status} | concept={concept:30s} verified={result} errors={err}")

print("\ntree-sitter setup OK")

In [ ]:
# Cell 6 – DOMAINS (5 domains with Rust-typed mock data)

DOMAINS = {
    "finance": {
        "var_names": {
            "item": "transaction", "func": "audit_record",
            "struct_name": "Portfolio", "method": "calculate_returns",
            "value": "balance", "key": "ticker", "module": "accounting",
            "enum_name": "Currency", "trait_name": "Auditable",
            "variant_a": "Usd", "variant_b": "Eur",
        },
        "mock_data": {
            "i32": "42500_i32", "i64": "42500_i64", "f64": "1050.50_f64",
            "u8": "42_u8", "u32": "150_u32", "u64": "150_u64",
            "usize": "150_usize", "isize": "42_isize",
            "bool": "true", "char": "'$'",
            "str_literal": '"USD-2024-Q3-REPORT"',
            "string_expr": 'String::from("USD-2024-Q3")',
            "vec_i32": "vec![1050, -20, 400, 88]",
            "vec_f64": "vec![1050.50, -20.0, 400.25, 88.10]",
            "vec_str": 'vec!["AAPL", "GOOG", "MSFT"]',
            "option_i32": "Some(42500_i32)",
            "result_i32": "Ok(42500_i32)",
            "hashmap": 'HashMap::from([("AAPL", 178.50), ("GOOG", 140.20)])',
        },
    },
    "biology": {
        "var_names": {
            "item": "genome_sequence", "func": "analyze_genome",
            "struct_name": "SequenceAnalyzer", "method": "run_alignment",
            "value": "mutation_rate", "key": "gene_id", "module": "genomics",
            "enum_name": "Nucleotide", "trait_name": "Sequenceable",
            "variant_a": "Adenine", "variant_b": "Thymine",
        },
        "mock_data": {
            "i32": "43044295_i32", "i64": "43044295_i64", "f64": "0.95_f64",
            "u8": "17_u8", "u32": "17_u32", "u64": "43044295_u64",
            "usize": "17_usize", "isize": "17_isize",
            "bool": "false", "char": "'A'",
            "str_literal": '"BRCA1-VARIANT-P53"',
            "string_expr": 'String::from("BRCA1")',
            "vec_i32": "vec![43044295, 43125483, 43170245]",
            "vec_f64": "vec![0.95, 0.87, 0.99, 0.42]",
            "vec_str": 'vec!["ACTG", "GCTA", "CGAT"]',
            "option_i32": "Some(43044295_i32)",
            "result_i32": "Ok(43044295_i32)",
            "hashmap": 'HashMap::from([("BRCA1", 17), ("TP53", 17)])',
        },
    },
    "gaming": {
        "var_names": {
            "item": "player_entry", "func": "update_leaderboard",
            "struct_name": "GameEngine", "method": "spawn_entity",
            "value": "hit_points", "key": "player_id", "module": "engine",
            "enum_name": "CharacterClass", "trait_name": "Spawnable",
            "variant_a": "Warrior", "variant_b": "Mage",
        },
        "mock_data": {
            "i32": "9500_i32", "i64": "9500_i64", "f64": "1.5_f64",
            "u8": "100_u8", "u32": "9500_u32", "u64": "9500_u64",
            "usize": "42_usize", "isize": "42_isize",
            "bool": "true", "char": "'W'",
            "str_literal": '"LEVEL_42_BOSS"',
            "string_expr": 'String::from("LEVEL_42")',
            "vec_i32": "vec![9500, 8700, 12400, 6300]",
            "vec_f64": "vec![1.5, 2.0, 0.8, 3.2]",
            "vec_str": 'vec!["warrior", "mage", "rogue"]',
            "option_i32": "Some(9500_i32)",
            "result_i32": "Ok(9500_i32)",
            "hashmap": 'HashMap::from([("health", 100), ("mana", 50)])',
        },
    },
    "physics": {
        "var_names": {
            "item": "reading", "func": "compute_trajectory",
            "struct_name": "SimulationRunner", "method": "step_forward",
            "value": "velocity", "key": "particle_id", "module": "simulation",
            "enum_name": "Particle", "trait_name": "Simulatable",
            "variant_a": "Proton", "variant_b": "Electron",
        },
        "mock_data": {
            "i32": "299792458_i32", "i64": "299792458_i64", "f64": "9.81_f64",
            "u8": "42_u8", "u32": "299792458_u32", "u64": "299792458_u64",
            "usize": "42_usize", "isize": "42_isize",
            "bool": "false", "char": "'e'",
            "str_literal": '"MUON_DECAY_EVENT"',
            "string_expr": 'String::from("MUON_DECAY")',
            "vec_i32": "vec![981, 300000000, 667]",
            "vec_f64": "vec![9.81, 3.0e8, 6.674e-11]",
            "vec_str": 'vec!["proton", "neutron", "electron"]',
            "option_i32": "Some(299792458_i32)",
            "result_i32": "Ok(299792458_i32)",
            "hashmap": 'HashMap::from([("mass", 938), ("charge", 1)])',
        },
    },
    "ecommerce": {
        "var_names": {
            "item": "order_item", "func": "process_checkout",
            "struct_name": "OrderProcessor", "method": "apply_discount",
            "value": "total_price", "key": "sku", "module": "checkout",
            "enum_name": "OrderStatus", "trait_name": "Processable",
            "variant_a": "Shipped", "variant_b": "Pending",
        },
        "mock_data": {
            "i32": "142_i32", "i64": "142_i64", "f64": "29.99_f64",
            "u8": "3_u8", "u32": "142_u32", "u64": "142_u64",
            "usize": "3_usize", "isize": "3_isize",
            "bool": "true", "char": "'#'",
            "str_literal": '"ORDER-2024-XK9001"',
            "string_expr": 'String::from("ORDER-2024")',
            "vec_i32": "vec![2999, 1550, 8900, 499]",
            "vec_f64": "vec![29.99, 15.50, 89.00, 4.99]",
            "vec_str": 'vec!["WDG-4420", "SKU-1001", "ITM-7788"]',
            "option_i32": "Some(142_i32)",
            "result_i32": "Ok(142_i32)",
            "hashmap": 'HashMap::from([("WDG-4420", 2999), ("SKU-1001", 1550)])',
        },
    },
}

print(f"Domains: {list(DOMAINS.keys())}")

In [ ]:
# Cell 7 – Concept space: families, compatibility, sparse matrix

RUST_CONSTRUCT_FAMILIES = {
    "iteration":    ["for_expression", "while_expression", "loop_expression"],
    "branching":    ["if_expression", "match_expression"],
    "function":     ["function_item", "closure_expression"],
    "binding":      ["let_declaration", "let_mut", "const_item", "static_item"],
    "struct_enum":  ["struct_item", "enum_item", "type_alias"],
    "impl_trait":   ["impl_item", "trait_item"],
    "module":       ["use_declaration", "mod_item"],
    "error":        ["macro_invocation_question_mark"],
    "flow":         ["return_expression", "break_expression", "continue_expression"],
    "async":        ["async_block", "await_expression"],
    "unsafe":       ["unsafe_block"],
    "ownership":    ["reference_expression", "mutable_reference", "dereference"],
    "lifetime":     ["lifetime"],
    "macro":        ["macro_invocation"],
    "attribute":    ["attribute_item"],
}

RUST_OBJECT_FAMILIES = {
    "primitives":           ["i32", "i64", "f64", "u8", "u32", "u64", "usize",
                             "isize", "bool", "char"],
    "prelude_types":        ["Vec", "String", "Box", "Option", "Result"],
    "prelude_enum_variants":["Some", "None", "Ok", "Err"],
    "prelude_traits":       ["Clone", "Copy", "Default", "Debug", "Display",
                             "Iterator", "IntoIterator", "From", "Into",
                             "TryFrom", "TryInto", "AsRef", "AsMut",
                             "PartialEq", "Eq", "PartialOrd", "Ord",
                             "Send", "Sync", "Drop", "Sized",
                             "Fn", "FnMut", "FnOnce", "ToString"],
    "common_std":           ["HashMap", "HashSet", "BTreeMap", "VecDeque",
                             "Rc", "Arc", "Mutex", "Cell", "RefCell"],
}

RUST_COMPATIBILITY = {
    "iteration":    ["primitives", "prelude_types", "common_std"],
    "branching":    ["primitives", "prelude_types", "prelude_enum_variants"],
    "function":     ["primitives", "prelude_types", "prelude_traits"],
    "binding":      ["primitives", "prelude_types", "common_std"],
    "struct_enum":  ["primitives", "prelude_types", "prelude_traits"],
    "impl_trait":   ["prelude_traits", "prelude_types"],
    "module":       [],
    "error":        ["prelude_enum_variants", "prelude_types"],
    "flow":         ["primitives", "prelude_types"],
    "async":        ["prelude_types", "prelude_traits"],
    "unsafe":       ["primitives", "prelude_types"],
    "ownership":    ["primitives", "prelude_types", "common_std"],
    "lifetime":     ["primitives", "prelude_types"],
    "macro":        ["primitives", "prelude_types"],
    "attribute":    ["prelude_traits"],
}

CONSTRUCT_DISPLAY_NAMES = {
    "for_expression": "For", "while_expression": "While",
    "loop_expression": "Loop", "if_expression": "If",
    "match_expression": "Match", "function_item": "Fn",
    "closure_expression": "Closure", "let_declaration": "Let",
    "let_mut": "LetMut", "const_item": "Const",
    "static_item": "Static", "struct_item": "Struct",
    "enum_item": "Enum", "type_alias": "TypeAlias",
    "impl_item": "Impl", "trait_item": "Trait",
    "use_declaration": "Use", "mod_item": "Mod",
    "return_expression": "Return", "break_expression": "Break",
    "continue_expression": "Continue", "async_block": "Async",
    "await_expression": "Await", "unsafe_block": "Unsafe",
    "reference_expression": "Ref", "mutable_reference": "RefMut",
    "dereference": "Deref", "lifetime": "Lifetime",
    "macro_invocation": "Macro", "attribute_item": "Attribute",
    "macro_invocation_question_mark": "QuestionMark",
}

ITEM_LEVEL = {"Fn", "Struct", "Enum", "Impl", "Trait", "Use", "Mod",
              "Const", "Static", "TypeAlias", "Attribute"}

STMT_LEVEL = {"For", "While", "Loop", "If", "Match", "Let", "LetMut",
              "Return", "Break", "Continue", "Async", "Await",
              "Unsafe", "Ref", "RefMut", "Deref", "Lifetime",
              "Macro", "Closure", "QuestionMark"}


def build_sparse_matrix(mode: str) -> list:
    """Build (construct, object) pairs based on compatibility."""
    all_constructs = set()
    for members in RUST_CONSTRUCT_FAMILIES.values():
        all_constructs.update(members)

    all_objects = set()
    for members in RUST_OBJECT_FAMILIES.values():
        all_objects.update(members)

    pairs = []
    for construct_fam, object_fams in RUST_COMPATIBILITY.items():
        constructs = [c for c in RUST_CONSTRUCT_FAMILIES[construct_fam] if c in all_constructs]

        if not object_fams:
            for c in constructs:
                pairs.append((c, "_isolated_"))
            continue

        allowed_objects = set()
        for of in object_fams:
            allowed_objects |= set(RUST_OBJECT_FAMILIES[of])
        allowed_objects &= all_objects

        for c in constructs:
            for obj in sorted(allowed_objects):
                pairs.append((c, obj))

    if mode == "test":
        test_constructs = {"for_expression", "if_expression", "let_declaration",
                           "function_item", "struct_item"}
        test_objects = {"i32", "Vec", "String", "Option", "HashMap"}
        pairs = [(c, o) for c, o in pairs
                 if c in test_constructs and (o in test_objects or o == "_isolated_")]
    elif mode == "small":
        small_constructs = {
            "for_expression", "while_expression", "loop_expression",
            "if_expression", "match_expression",
            "function_item", "closure_expression",
            "let_declaration", "let_mut", "const_item",
            "struct_item", "enum_item",
            "impl_item", "trait_item",
            "use_declaration",
            "return_expression", "break_expression", "continue_expression",
            "unsafe_block", "reference_expression", "macro_invocation",
            "attribute_item",
        }
        pairs = [(c, o) for c, o in pairs
                 if c in small_constructs or o == "_isolated_"]

    return pairs


concept_pairs = build_sparse_matrix(MODE)
print(f"Concept pairs: {len(concept_pairs)} (mode={MODE})")
print(f"Unique constructs: {len(set(c for c, _ in concept_pairs))}")
print(f"Unique objects: {len(set(o for _, o in concept_pairs))}")
print(f"Sample: {concept_pairs[:5]}")

In [ ]:
# Cell 8 – RustPromptGenerator class

PADDING_BEFORE = [
    "",
    "let _result: Option<i32> = None;",
    'println!("Starting process");',
    "let _status = true;",
    "let mut _counter: i32 = 0;",
]

PADDING_AFTER = [
    "",
    'println!("Done");',
    "let _status = false;",
    "_counter += 1;",
    "let _result: Option<i32> = None;",
]

WRAPPER_TYPES = ["bare_fn_main", "inside_fn", "inside_impl"]
WRAPPER_WEIGHTS = [0.50, 0.30, 0.20]


class RustPromptGenerator:

    def __init__(self, domains: dict):
        self.domains = domains
        self.domain_keys = list(domains.keys())

    def generate_batch(self, construct: str, obj: str, n: int = 150) -> list:
        results = []
        display = CONSTRUCT_DISPLAY_NAMES.get(construct, construct)
        for i in range(n):
            try:
                dk = self.domain_keys[i % len(self.domain_keys)]
                domain = self.domains[dk]
                d, m = domain["var_names"], domain["mock_data"]

                essence = self._build_essence(construct, obj, d, m)
                if essence is None:
                    continue

                pad_b = random.choice(PADDING_BEFORE)
                pad_a = random.choice(PADDING_AFTER)
                body = "\n".join(p for p in [pad_b, essence, pad_a] if p)

                wrapper = random.choices(WRAPPER_TYPES, weights=WRAPPER_WEIGHTS, k=1)[0]
                wrapped = self._apply_wrapper(body, construct, wrapper, d)

                if has_errors(wrapped):
                    continue

                verified = verify_concept(wrapped, construct)

                results.append({
                    "prompt_text": wrapped,
                    "tree_sitter_verified": verified,
                    "domain": dk,
                    "wrapper": wrapper,
                })
            except Exception as e:
                logger.debug(f"Error ({construct}, {obj}) #{i}: {e}")
                continue

        if len(results) == 0:
            logger.warning(f"ZERO prompts for ({construct}, {obj})")
        return results

    def _build_essence(self, construct, obj, d, m):
        if obj == "_isolated_":
            return self._isolated_tpl(construct, d, m)

        def mock(t=None):
            """Get mock data for a type. Falls back to i32."""
            if t:
                return m.get(t, m["i32"])
            # Map object name to mock key
            obj_to_mock = {
                "i32": "i32", "i64": "i64", "f64": "f64",
                "u8": "u8", "u32": "u32", "u64": "u64",
                "usize": "usize", "isize": "isize",
                "bool": "bool", "char": "char",
                "Vec": "vec_i32", "String": "string_expr",
                "Box": "i32", "Option": "option_i32",
                "Result": "result_i32",
                "HashMap": "hashmap", "HashSet": "vec_i32",
                "Some": "option_i32", "None": "option_i32",
                "Ok": "result_i32", "Err": "result_i32",
            }
            key = obj_to_mock.get(obj, "i32")
            return m.get(key, m["i32"])

        def type_annotation():
            """Get Rust type annotation for the current object."""
            simple = {"i32", "i64", "f64", "u8", "u32", "u64", "usize", "isize", "bool", "char"}
            if obj in simple:
                return obj
            type_map = {
                "Vec": "Vec<i32>", "String": "String",
                "Box": "Box<i32>", "Option": "Option<i32>",
                "Result": "Result<i32, String>",
                "HashMap": 'HashMap<&str, f64>',
                "HashSet": "HashSet<i32>",
                "BTreeMap": 'BTreeMap<&str, i32>',
                "VecDeque": "VecDeque<i32>",
                "Rc": "Rc<i32>", "Arc": "Arc<i32>",
                "Mutex": "Mutex<i32>",
                "Cell": "Cell<i32>", "RefCell": "RefCell<i32>",
                "Some": "Option<i32>", "None": "Option<i32>",
                "Ok": "Result<i32, String>", "Err": "Result<i32, String>",
            }
            return type_map.get(obj, "i32")

        def needs_use():
            """Return a use statement if the object needs one, else empty."""
            use_map = {
                "HashMap": "use std::collections::HashMap;",
                "HashSet": "use std::collections::HashSet;",
                "BTreeMap": "use std::collections::BTreeMap;",
                "VecDeque": "use std::collections::VecDeque;",
                "Rc": "use std::rc::Rc;",
                "Arc": "use std::sync::Arc;",
                "Mutex": "use std::sync::Mutex;",
                "Cell": "use std::cell::Cell;",
                "RefCell": "use std::cell::RefCell;",
            }
            return use_map.get(obj, "")

        def iterable_expr():
            """Produce an iterable binding for the current object."""
            if obj == "Vec":
                return f"let {d['value']}: Vec<i32> = {m['vec_i32']};"
            elif obj == "HashMap":
                return f"{needs_use()}\nlet {d['value']} = {m['hashmap']};"
            elif obj == "String":
                return f"let {d['value']} = {m['string_expr']};"
            elif obj in ("i32", "u32", "usize"):
                return f"let {d['value']}: Vec<{obj}> = vec![1, 2, 3, 4, 5];"
            else:
                return f"let {d['value']}: Vec<i32> = {m['vec_i32']};"

        T = {
            # ── Iteration ──
            "for_expression": lambda: f"""{iterable_expr()}
for {d['item']} in &{d['value']} {{
    println!("{{:?}}", {d['item']});
}}""",

            "while_expression": lambda: f"""let mut {d['value']}: i32 = {mock('i32')};
while {d['value']} > 0 {{
    println!("{{}}", {d['value']});
    {d['value']} -= 1;
}}""",

            "loop_expression": lambda: f"""let mut {d['value']}: i32 = {mock('i32')};
loop {{
    if {d['value']} <= 0 {{ break; }}
    {d['value']} -= 1;
}}""",

            # ── Branching ──
            "if_expression": lambda: f"""let {d['value']}: {type_annotation()} = {mock()};
if {d['value']} > 0 {{
    println!("positive");
}} else {{
    println!("non-positive");
}}""" if obj in ("i32","i64","f64","u32","u64","usize","isize","u8") else f"""let {d['value']}: {type_annotation()} = {mock()};
if {d['value']}.is_some() {{
    println!("has value");
}}""" if obj in ("Option","Some","None") else f"""let {d['value']} = {mock('bool')};
if {d['value']} {{
    println!("true branch");
}} else {{
    println!("false branch");
}}""",

            "match_expression": lambda: f"""let {d['value']}: Option<i32> = {m['option_i32']};
match {d['value']} {{
    Some(x) => println!("{{x}}"),
    None => println!("nothing"),
}}""" if obj in ("Option","Some","None") else f"""let {d['value']}: Result<i32, String> = {m['result_i32']};
match {d['value']} {{
    Ok(x) => println!("{{x}}"),
    Err(e) => println!("{{e}}"),
}}""" if obj in ("Result","Ok","Err") else f"""let {d['value']}: i32 = {mock('i32')};
match {d['value']} {{
    0 => println!("zero"),
    1..=100 => println!("small"),
    _ => println!("large"),
}}""",

            # ── Function ──
            "function_item": lambda: f"""fn {d['func']}({d['item']}: {type_annotation()}) -> {type_annotation()} {{
    {d['item']}
}}""",

            "closure_expression": lambda: f"""let {d['func']} = |{d['item']}: {type_annotation()}| -> {type_annotation()} {{
    {d['item']}
}};
let _result = {d['func']}({mock()});""",

            # ── Binding ──
            "let_declaration": lambda: f"let {d['value']}: {type_annotation()} = {mock()};",

            "let_mut": lambda: f"""let mut {d['value']}: {type_annotation()} = {mock()};
{d['value']} = {mock()};""",

            "const_item": lambda: f"const {d['key'].upper()}: {type_annotation()} = {mock()};",

            "static_item": lambda: f"static {d['key'].upper()}: {type_annotation()} = {mock()};",

            # ── Struct/Enum ──
            "struct_item": lambda: f"""struct {d['struct_name']} {{
    {d['value']}: {type_annotation()},
    {d['key']}: String,
}}""",

            "enum_item": lambda: f"""enum {d['enum_name']} {{
    {d['variant_a']}({type_annotation()}),
    {d['variant_b']},
}}""",

            "type_alias": lambda: f"type {d['struct_name']}Id = {type_annotation()};",

            # ── Impl/Trait ──
            "impl_item": lambda: f"""struct {d['struct_name']}Data;
impl {d['struct_name']}Data {{
    fn {d['method']}(&self) -> {type_annotation()} {{
        {mock()}
    }}
}}""",

            "trait_item": lambda: f"""trait {d['trait_name']} {{
    fn {d['method']}(&self) -> {type_annotation()};
}}""",

            # ── Flow ──
            "return_expression": lambda: f"""fn {d['func']}() -> {type_annotation()} {{
    let {d['value']}: {type_annotation()} = {mock()};
    return {d['value']};
}}""",

            "break_expression": lambda: f"""let mut {d['value']}: i32 = {mock('i32')};
loop {{
    if {d['value']} <= 0 {{
        break;
    }}
    {d['value']} -= 1;
}}""",

            "continue_expression": lambda: f"""{iterable_expr()}
for {d['item']} in &{d['value']} {{
    if *{d['item']} == 0 {{
        continue;
    }}
    println!("{{:?}}", {d['item']});
}}""",

            # ── Async ──
            "async_block": lambda: f"""let _future = async {{
    let {d['value']}: {type_annotation()} = {mock()};
    println!("{{:?}}", {d['value']});
}};""",

            "await_expression": lambda: f"""async fn {d['func']}() -> {type_annotation()} {{
    {mock()}
}}
async fn run() {{
    let _result = {d['func']}().await;
}}""",

            # ── Unsafe ──
            "unsafe_block": lambda: f"""let {d['value']}: {type_annotation()} = {mock()};
unsafe {{
    let _ptr = &{d['value']} as *const {type_annotation()};
    println!("{{:?}}", *_ptr);
}}""",

            # ── Ownership ──
            "reference_expression": lambda: f"""let {d['value']}: {type_annotation()} = {mock()};
let _ref = &{d['value']};
println!("{{:?}}", _ref);""",

            "mutable_reference": lambda: f"""let mut {d['value']}: {type_annotation()} = {mock()};
let _ref = &mut {d['value']};
println!("{{:?}}", _ref);""",

            "dereference": lambda: f"""let {d['value']}: {type_annotation()} = {mock()};
let _ref = &{d['value']};
let _deref = *_ref;""",

            # ── Lifetime ──
            "lifetime": lambda: f"""fn {d['func']}<'a>({d['item']}: &'a {type_annotation()}) -> &'a {type_annotation()} {{
    {d['item']}
}}""",

            # ── Macro ──
            "macro_invocation": lambda: f"""let {d['value']}: {type_annotation()} = {mock()};
println!("{{:?}}", {d['value']});""",

            # ── Attribute ──
            "attribute_item": lambda: f"""#[derive({obj})]
struct {d['struct_name']}Data {{
    {d['value']}: i32,
}}""" if obj in ("Clone","Copy","Default","Debug","PartialEq","Eq","PartialOrd","Ord") else f"""#[allow(dead_code)]
struct {d['struct_name']}Data {{
    {d['value']}: i32,
}}""",

            # ── Error (? operator) ──
            "macro_invocation_question_mark": lambda: f"""fn {d['func']}() -> Result<{type_annotation()}, Box<dyn std::error::Error>> {{
    let {d['value']}: {type_annotation()} = {mock()};
    let _text = {d['value']}.to_string();
    let _parsed: i32 = _text.parse()?;
    Ok({d['value']})
}}""",
        }

        builder = T.get(construct)
        if builder is None:
            return None
        try:
            return builder()
        except Exception:
            return None

    def _isolated_tpl(self, construct, d, m):
        """Templates for constructs with no paired object."""
        T = {
            "use_declaration": lambda: "use std::collections::HashMap;",
            "mod_item": lambda: f"mod {d['module']} {{
    pub fn init() {{}}
}}",
        }
        builder = T.get(construct)
        return builder() if builder else None

    def _apply_wrapper(self, body, construct, wrapper, d):
        """Wrap body in appropriate Rust scope."""
        display = CONSTRUCT_DISPLAY_NAMES.get(construct, construct)

        if display in ITEM_LEVEL:
            # Item-level constructs go at module level
            if wrapper == "bare_fn_main":
                return f"{body}\n\nfn main() {{}}"
            elif wrapper == "inside_fn":
                return f"{body}\n\nfn {d['func']}_wrapper() {{}}"
            else:
                return f"{body}\n\nfn main() {{}}"
        else:
            # Statement-level constructs go inside a function body
            indented = textwrap.indent(body, "    ")
            if wrapper == "bare_fn_main":
                return f"fn main() {{\n{indented}\n}}"
            elif wrapper == "inside_fn":
                return f"fn {d['func']}_wrapper() {{\n{indented}\n}}"
            elif wrapper == "inside_impl":
                return (f"struct {d['struct_name']}Ctx;\n"
                        f"impl {d['struct_name']}Ctx {{\n"
                        f"    fn {d['method']}_run(&self) {{\n"
                        f"{textwrap.indent(body, '        ')}\n"
                        f"    }}\n}}")
        return body


# Smoke test
gen = RustPromptGenerator(DOMAINS)
for construct in ["for_expression", "if_expression", "let_declaration", "function_item", "struct_item"]:
    batch = gen.generate_batch(construct, "i32", n=5)
    ok = sum(1 for p in batch if p["tree_sitter_verified"])
    print(f"  {construct:30s} generated={len(batch)} verified={ok}")
    if batch:
        print(f"    Sample: {batch[0]['prompt_text'][:80]}...")

print("\nRustPromptGenerator OK")

In [ ]:
# Cell 9 – PerplexityFilter (identical to Python 1A)
from transformers import AutoTokenizer, AutoModelForCausalLM


class PerplexityFilter:

    def __init__(self, model_name: str, device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Loading {model_name} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        ).eval()
        logger.info("Model loaded.")

    def score_prompt(self, prompt_text: str) -> tuple:
        with torch.no_grad():
            input_ids = self.tokenizer(
                prompt_text, return_tensors="pt", truncation=True,
                max_length=512, add_special_tokens=False
            )["input_ids"].to(self.device)
            token_length = input_ids.shape[1]
            if token_length < 2:
                return 50.0, token_length
            logits = self.model(input_ids).logits
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            ).item()
        return loss, token_length

    def filter_batch(self, prompts, top_k=100, catastrophic_threshold=3.0):
        if not prompts:
            return []
        for p in prompts:
            loss, tok_len = self.score_prompt(p["prompt_text"])
            p["sequence_loss"] = loss
            p["token_length"] = tok_len
        avg_loss = np.mean([p["sequence_loss"] for p in prompts])
        if avg_loss > catastrophic_threshold:
            logger.warning(f"Avg loss {avg_loss:.2f} > {catastrophic_threshold}. Discarding batch.")
            return []
        return prompts[:top_k]


pfilter = PerplexityFilter(MODEL_ID)
print("PerplexityFilter ready.")

In [ ]:
# Cell 10 – Pipeline orchestration

def run_pipeline(concept_pairs, generator, pfilter, n_generate, n_keep,
                 checkpoint_dir, checkpoint_every=50, run_name=None,
                 catastrophic_threshold=10.0):

    if run_name is None:
        run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")

    all_rows = []
    stats = {
        "run_name": run_name, "mode": MODE,
        "n_constructs": len(set(c for c, _ in concept_pairs)),
        "n_objects": len(set(o for _, o in concept_pairs)),
        "total_pairs": len(concept_pairs),
        "n_generate": n_generate, "n_keep": n_keep,
        "successful_pairs": 0, "failed_pairs": 0,
        "empty_gen_pairs": [], "catastrophic_pairs": [],
        "total_prompts": 0,
    }

    for idx, (construct, obj) in enumerate(tqdm(concept_pairs, desc="Pairs")):
        raw = generator.generate_batch(construct, obj, n=n_generate)
        if not raw:
            stats["failed_pairs"] += 1
            stats["empty_gen_pairs"].append([construct, obj])
            continue

        filtered = pfilter.filter_batch(
            raw, top_k=n_keep, catastrophic_threshold=catastrophic_threshold
        )
        if not filtered:
            stats["failed_pairs"] += 1
            stats["catastrophic_pairs"].append([construct, obj])
            continue

        display = CONSTRUCT_DISPLAY_NAMES.get(construct, construct)
        for var_id, p in enumerate(filtered):
            all_rows.append({
                "construct": display, "object": obj,
                "variation_id": var_id, "prompt_text": p["prompt_text"],
                "sequence_loss": p["sequence_loss"],
                "token_length": p["token_length"],
                "tree_sitter_verified": p.get("tree_sitter_verified", False),
            })
        stats["successful_pairs"] += 1
        stats["total_prompts"] += len(filtered)

        if (idx + 1) % checkpoint_every == 0:
            ckpt = pd.DataFrame(all_rows)
            path = os.path.join(checkpoint_dir, f"{PREFIX}{run_name}_ckpt_{idx+1}.parquet")
            ckpt.to_parquet(path, index=False)
            logger.info(f"Checkpoint: {path} ({len(ckpt)} rows)")

    df = pd.DataFrame(all_rows)
    final_path = os.path.join(checkpoint_dir, OUTPUT_FILE)
    df.to_parquet(final_path, index=False)

    stats_path = os.path.join(checkpoint_dir, STATS_FILE)
    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"\n{'='*60}")
    print(f"DONE")
    print(f"  Pairs: {stats['successful_pairs']}/{stats['total_pairs']} succeeded")
    print(f"  Prompts: {stats['total_prompts']}")
    print(f"  Output: {final_path}")
    print(f"{'='*60}")
    return df

In [ ]:
# Cell 11 – Execute pipeline
generator = RustPromptGenerator(DOMAINS)

df = run_pipeline(
    concept_pairs=concept_pairs,
    generator=generator, pfilter=pfilter,
    n_generate=N_GENERATE, n_keep=N_KEEP,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=CHECKPOINT_EVERY,
    run_name=run_name,
    catastrophic_threshold=CATASTROPHIC_THRESHOLD,
)
df.head(10)

In [ ]:
# Cell 12 – Quality report
print("=" * 60)
print("QUALITY REPORT")
print("=" * 60)

if df.empty:
    print("\nDataFrame is empty -- no prompts survived the pipeline.")
    print(f"   Pairs attempted : {len(concept_pairs)}")
    stats_path = os.path.join(CHECKPOINT_DIR, STATS_FILE)
    if os.path.exists(stats_path):
        with open(stats_path) as f:
            stats = json.load(f)
        print(f"   successful_pairs   : {stats.get('successful_pairs')}")
        print(f"   failed_pairs       : {stats.get('failed_pairs')}")
        print(f"   empty_gen_pairs    : {stats.get('empty_gen_pairs')}")
        print(f"   catastrophic_pairs : {stats.get('catastrophic_pairs')}")
else:
    pair_counts = df.groupby(["construct", "object"]).size().reset_index(name="count")
    total_attempted = len(concept_pairs)
    print(f"\n1. PAIR COVERAGE: {len(pair_counts)} / {total_attempted} "
          f"({100*len(pair_counts)/total_attempted:.0f}%)")

    min_expected = int(N_KEEP * 0.8)
    good = pair_counts[pair_counts["count"] >= min_expected]
    print(f"   Pairs with >={min_expected} prompts: {len(good)}")

    verified_rate = df["tree_sitter_verified"].mean() * 100
    print(f"\n2. TREE-SITTER VERIFIED: {verified_rate:.1f}%")

    print(f"\n3. LOSS DISTRIBUTION")
    for stat, val in [("Mean", df["sequence_loss"].mean()),
                      ("Median", df["sequence_loss"].median()),
                      ("Std", df["sequence_loss"].std()),
                      ("Min", df["sequence_loss"].min()),
                      ("Max", df["sequence_loss"].max())]:
        print(f"   {stat}: {val:.3f}")

    print(f"\n4. TOKEN LENGTH: mean={df['token_length'].mean():.0f}, "
          f"range=[{df['token_length'].min()}, {df['token_length'].max()}]")

    print(f"\n5. SAMPLE PROMPTS")
    print("-" * 60)
    n_samples = min(5, len(pair_counts))
    for _, row in pair_counts.sample(n_samples, random_state=0).iterrows():
        sub = df[(df["construct"] == row["construct"]) & (df["object"] == row["object"])]
        s = sub.iloc[0]
        print(f"\n  pair : ({s['construct']}, {s['object']})")
        print(f"  loss : {s['sequence_loss']:.4f}  |  tokens: {s['token_length']}  |  verified: {s['tree_sitter_verified']}")
        print(f"  prompt:\n{s['prompt_text']}")
        print("-" * 60)

In [ ]:
# Cell 13 – Loss histogram
import matplotlib.pyplot as plt

if df.empty:
    print("No data to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(df["sequence_loss"], bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Sequence Loss")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Loss Distribution (All Prompts)")
    axes[0].axvline(df["sequence_loss"].median(), color="red", linestyle="--",
                    label=f"Median: {df['sequence_loss'].median():.2f}")
    axes[0].legend()

    pair_loss = df.groupby(["construct", "object"])["sequence_loss"].mean()
    axes[1].hist(pair_loss, bins=20, edgecolor="black", alpha=0.7, color="orange")
    axes[1].set_xlabel("Mean Loss per Pair")
    axes[1].set_title("Per-Pair Mean Loss")

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 14 – Variance check
print("VARIANCE CHECK")
print("=" * 60)

if df.empty:
    print("No data.")
else:
    sc, so = df["construct"].iloc[0], df["object"].iloc[0]
    sub = df[(df["construct"] == sc) & (df["object"] == so)]
    print(f"Pair: ({sc}, {so}) -- {len(sub)} variations\n")

    for i, (_, r) in enumerate(sub.head(5).iterrows()):
        print(f"--- Var {r['variation_id']}  loss={r['sequence_loss']:.4f}  tokens={r['token_length']} ---")
        print(r["prompt_text"])
        print()

    firsts = [p.split("\n")[0] for p in sub["prompt_text"]]
    print(f"Unique first lines: {len(set(firsts))}/{len(firsts)}")

In [ ]:
# Cell 15 – Cleanup
import glob
ckpts = glob.glob(os.path.join(CHECKPOINT_DIR, f"{PREFIX}{run_name}_ckpt_*.parquet"))
print(f"Found {len(ckpts)} checkpoint files.")
# Uncomment to delete:
# for f in ckpts: os.remove(f)
# print("Deleted.")

try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab -- skipping unassign.")